# 03 · Preprocesamiento del texto y vectorización TF-IDF

Corresponde a las **secciones 4 (Preprocesamiento) y 5 (Vectorización)** del protocolo. Las reseñas están en **inglés**, por lo que
se usan stopwords y lematizador de inglés (NLTK + WordNet).

Pasos del preprocesamiento (función `utils.preprocesar_texto`):

| Paso | Qué hace | Ejemplo |
|---|---|---|
| Normalización Unicode | quita acentos, emojis y caracteres raros | `café 😀` → `cafe` |
| Minúsculas | unifica mayúsculas | `GREAT` → `great` |
| HTML y URLs | elimina `<br />`, `&amp;`, enlaces | |
| Negaciones | expande contracciones para no perderlas | `doesn't` → `does not` |
| Puntuación, números y caracteres especiales | se eliminan | `damaged!!!` → `damaged` |
| Letras repetidas | reduce alargamientos | `soooo` → `soo` |
| Tokenización | separa en palabras (`word_tokenize`) | |
| Lematización con etiqueta gramatical | lleva cada palabra a su forma base | `arrived` → `arrive` |
| Stopwords | quita palabras vacías **conservando negaciones** (`not, no, never, nor...`) | `the, and` se van; `not` se queda |
| Espacios | se eliminan espacios redundantes | |

**¿Por qué conservar las negaciones?** Porque cambian el significado: *"not working"* (defecto) frente a *"working"* (elogio).
La lista de stopwords de NLTK incluye `not` y `no`, así que las retiramos de esa lista.

In [1]:
import time
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nltk import word_tokenize, pos_tag

import utils
from utils import CLASES, ARCHIVO_ETIQUETADO, ARCHIVO_PROCESADO, RESULTADOS, PARAMETROS_TFIDF

SALIDA = RESULTADOS / "preprocesamiento"
SALIDA.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 150)

utils.preparar_nltk()  # descarga stopwords, WordNet, punkt y el etiquetador gramatical si faltan
print("Stopwords de NLTK usadas:", len(utils._STOPWORDS))
print("Negaciones que se CONSERVAN:", sorted(utils.NEGACIONES_CONSERVAR))

Stopwords de NLTK usadas: 195
Negaciones que se CONSERVAN: ['neither', 'never', 'no', 'nobody', 'none', 'nor', 'not', 'nothing', 'nowhere', 'without']


## 1. Ejemplo del protocolo, paso a paso

Texto original: **"The package arrived late and damaged!!!"** → texto procesado esperado: **"package arrive late damage"**.

In [2]:
original = "The package arrived late and damaged!!!"

limpio = utils.limpiar_texto(original)
tokens = word_tokenize(limpio)
etiquetas = pos_tag(tokens)
lemas = [utils._LEMATIZADOR.lemmatize(t, utils._etiqueta_wordnet(p)) for t, p in etiquetas]
sin_stopwords = [l for l in lemas if l not in utils._STOPWORDS and len(l) > 1]

print("1. Original:              ", original)
print("2. Limpieza:              ", limpio)
print("3. Tokens:                ", tokens)
print("4. Etiquetas gramaticales:", etiquetas)
print("5. Lemas:                 ", lemas)
print("6. Sin stopwords:         ", sin_stopwords)
procesado = utils.preprocesar_texto(original)
print("\nResultado final:", repr(procesado))
assert procesado == "package arrive late damage", "El preprocesamiento no reproduce el ejemplo del protocolo"
print("✔ Coincide con el ejemplo del protocolo")

1. Original:               The package arrived late and damaged!!!
2. Limpieza:               the package arrived late and damaged
3. Tokens:                 ['the', 'package', 'arrived', 'late', 'and', 'damaged']
4. Etiquetas gramaticales: [('the', 'DT'), ('package', 'NN'), ('arrived', 'VBD'), ('late', 'JJ'), ('and', 'CC'), ('damaged', 'VBD')]
5. Lemas:                  ['the', 'package', 'arrive', 'late', 'and', 'damage']
6. Sin stopwords:          ['package', 'arrive', 'late', 'damage']

Resultado final: 'package arrive late damage'
✔ Coincide con el ejemplo del protocolo


In [3]:
# Más ejemplos: negaciones, HTML, URL, letras repetidas y contracciones
pruebas = [
    "It doesn't work &amp; the lid is sooooo flimsy.<br />Won't buy again: https://example.com",
    "Not working after 2 weeks... NEVER again!!",
    "No leaks, no smell, and it's not cheap-looking at all :)",
    "Fast shipping 👍 but the handles broke off immediately",
]
pd.DataFrame({"original": pruebas, "procesado": [utils.preprocesar_texto(t) for t in pruebas]})

,original,procesado
0,It doesn't work &amp; the lid is sooooo flimsy.<br />Won't buy again: https://example.com,not work lid soo flimsy not buy
1,Not working after 2 weeks... NEVER again!!,not work week never
2,"No leaks, no smell, and it's not cheap-looking at all :)",no leaks no smell not cheap look
3,Fast shipping 👍 but the handles broke off immediately,fast shipping handle break immediately


## 2. Preprocesar todo el corpus

El preprocesamiento se aplica reseña por reseña y **no aprende nada de los datos** (no hay ajuste), por lo que puede aplicarse
antes de dividir sin fuga de información. Tarda unos minutos por el etiquetado gramatical.

In [4]:
df = pd.read_csv(ARCHIVO_ETIQUETADO, encoding="utf-8-sig")

inicio = time.perf_counter()
df["texto_procesado"] = df["text"].apply(utils.preprocesar_texto)
print(f"Preprocesamiento de {len(df)} reseñas: {time.perf_counter() - inicio:.1f} s")

df["n_tokens_procesados"] = df["texto_procesado"].str.split().str.len().fillna(0).astype(int)
print("Reseñas que quedaron vacías tras el preprocesamiento:", int((df["n_tokens_procesados"] == 0).sum()))
print("\nPalabras por reseña antes vs. después:")
display(pd.DataFrame({"original": df["n_palabras"].describe(), "procesado": df["n_tokens_procesados"].describe()}).round(1))
df[["text", "texto_procesado", "etiqueta_reglas"]].sample(5, random_state=utils.SEMILLA)

Preprocesamiento de 14826 reseñas: 247.4 s


Reseñas que quedaron vacías tras el preprocesamiento: 0

Palabras por reseña antes vs. después:


,original,procesado
count,14826.0,14826.0
mean,70.5,35.9
std,91.1,44.8
min,4.0,2.0
25%,20.0,12.0
50%,43.0,22.0
75%,84.0,42.0
max,1797.0,867.0


,text,texto_procesado,etiqueta_reglas
2609,"Flattened very quickly , not supportive at all.. Returned this pillow because it was a big disappointment. It flattened out quickly and I am not a...",flatten quickly not supportive return pillow big disappointment flatten quickly not heavy person think go support low back lumbar area nearly usel...,Producto
10050,Friend turned 70 - she loves it. Cute!,friend turn love cute,Sentimiento
6127,Nice product. Wish the pockets where a little wider and not so deep. My kindle and remotes sinks in and makes it hard to dig them out.,nice product wish pocket little wider not deep kindle remotes sink make hard dig,Sentimiento
7636,"ordered one a few weeks before which was awesome, ordered this one from the same supplier but .... Not the quality I had hoped, ordered one a few ...",order one week awesome order one supplier not quality hop order one week awesome order one supplier get different brand not happy,Sentimiento
10210,Nice. Used this Lamb's Ear Leaf on my lariat wreath home decor. Sold in one day! River Run Rustic Wood Crafts,nice use lamb ear leaf lariat wreath home decor sell one day river run rustic wood craft,Sentimiento


In [5]:
df.to_csv(ARCHIVO_PROCESADO, index=False, encoding="utf-8-sig")
print("Guardado:", ARCHIVO_PROCESADO.relative_to(utils.BASE), df.shape)

Guardado: datos\resenas_procesadas.csv (14826, 13)


## 3. Vectorización TF-IDF

**TF-IDF** representa cada reseña como un vector: el peso de una palabra crece si aparece mucho en esa reseña (TF) y baja si
aparece en casi todas las reseñas (IDF). Parámetros usados (iguales para los 4 modelos, definidos en `utils.PARAMETROS_TFIDF`):

- `ngram_range=(1, 2)`: unigramas y bigramas, para capturar expresiones como *"not work"* o *"arrive late"*.
- `min_df=3`: el término debe aparecer en al menos 3 reseñas (descarta errores de dedo y palabras rarísimas).
- `max_df=0.90`: descarta términos que aparecen en más del 90 % de las reseñas.
- `max_features=20000`: vocabulario acotado para controlar memoria y tiempo.
- `sublinear_tf=True`: usa `1 + log(tf)` para que una palabra repetida muchas veces no domine.

**Evitar fuga de datos:** el vectorizador se ajusta **solo con entrenamiento**. En los notebooks de modelos va dentro de un
`Pipeline` de scikit-learn, así que en cada pliegue de la validación cruzada también se ajusta solo con la parte de entrenamiento
de ese pliegue. Aquí lo ajustamos una vez solo para explorar la matriz resultante.

**Nunca** se usan `rating` ni `polarity` como características: el modelo solo ve el texto.

In [6]:
print("Parámetros TF-IDF:", PARAMETROS_TFIDF)
entrenamiento = df[df["particion"] == "entrenamiento"]
prueba = df[df["particion"] == "prueba"]

vectorizador = utils.crear_vectorizador()
X_train = vectorizador.fit_transform(entrenamiento["texto_procesado"])  # fit SOLO con entrenamiento
X_test = vectorizador.transform(prueba["texto_procesado"])               # prueba solo se transforma

vocabulario = vectorizador.get_feature_names_out()
n_bigramas = sum(" " in t for t in vocabulario)
print("Matriz de entrenamiento:", X_train.shape, "| Matriz de prueba:", X_test.shape)
print(f"Vocabulario: {len(vocabulario)} términos ({len(vocabulario) - n_bigramas} unigramas, {n_bigramas} bigramas)")
print(f"Densidad de la matriz de entrenamiento: {100 * X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.3f} % de celdas distintas de cero")

Parámetros TF-IDF: {'ngram_range': (1, 2), 'min_df': 3, 'max_df': 0.9, 'max_features': 20000, 'sublinear_tf': True}


Matriz de entrenamiento: (11860, 20000) | Matriz de prueba: (2966, 20000)
Vocabulario: 20000 términos (5247 unigramas, 14753 bigramas)
Densidad de la matriz de entrenamiento: 0.193 % de celdas distintas de cero


### Términos más característicos de cada etiqueta

Para cada clase calculamos el TF-IDF promedio de cada término en esa clase menos el promedio en las otras dos
(solo con entrenamiento). Es una primera mirada a lo que los modelos podrán aprender.

In [7]:
y_train = entrenamiento["etiqueta_reglas"].values
caracteristicos = {}
for clase in CLASES:
    en_clase = np.asarray(X_train[y_train == clase].mean(axis=0)).ravel()
    fuera = np.asarray(X_train[y_train != clase].mean(axis=0)).ravel()
    orden = np.argsort(en_clase - fuera)[::-1][:20]
    caracteristicos[clase] = vocabulario[orden]
caracteristicos = pd.DataFrame(caracteristicos)
caracteristicos.to_csv(SALIDA / "terminos_caracteristicos_por_clase.csv", index=False, encoding="utf-8-sig")
caracteristicos

,Envío,Producto,Sentimiento
0,arrive,not,love
1,broken,break,great
2,arrive broken,not work,perfect
3,come,smell,nice
4,damage,flimsy,beautiful
5,break,cheap,five star
6,come broken,thin,five
7,packaging,work,color
8,receive,poor,gift
9,package,not fit,cute


### Conclusión

El texto quedó normalizado y lematizado, con las negaciones conservadas, y guardado en `datos/resenas_procesadas.csv`
(columna `texto_procesado`). Los términos característicos coinciden con lo esperado: palabras de transporte para Envío,
defectos para Producto y opiniones generales para Sentimiento. Como las etiquetas se generaron con reglas léxicas, es natural
que varios de estos términos sean las propias palabras de las reglas; esto se discute en el análisis de errores y en la validación manual.